[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB03a_pytorch_taster.ipynb)

# NB03a · PyTorch taster: tensors, autograd, one learning step
**Artificial Intelligence in Medical Imaging** · Day 1 · Lab 3 (30 min)

| | |
|---|---|
| Runtime | CPU is enough |
| Data | synthetic points; a peek at PneumoniaMNIST (28 px, CC BY 4.0) |
| Checkpoints | tensor shapes · gradient equals the hand formula · loss goes down |
| Tiers | **Core** = TODO cells · **Plus** = learning rate experiment · **Challenge** = the same fit with `nn.Module` and `torch.optim` |

**Learning goals.** (1) Create tensors and read shape, dtype, device. (2) Let autograd compute a gradient and check it by hand. (3) Write a gradient-descent loop and see the loss fall. (4) Know what a `Dataset` and `DataLoader` give you.

In [ ]:
%pip install -q "medmnist==3.0.2"
import numpy as np, matplotlib.pyplot as plt, torch, random, os
def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)
from pathlib import Path
Path("data").mkdir(exist_ok=True)
print("torch", torch.__version__)

## 1. Tensors

**Predict:** what is the shape of a batch of 16 greyscale 28 × 28 images in PyTorch? (PyTorch puts channels first: `N, C, H, W`.)

**TODO 1.** Make a random batch and print shape, dtype and number of elements.

In [ ]:
# TODO 1: a random tensor with shape (16, 1, 28, 28)  (hint: torch.rand)
batch = None
print(batch.shape, batch.dtype, batch.numel())

In [ ]:
assert batch.shape == (16, 1, 28, 28) and batch.dtype == torch.float32
x = batch.flatten(1)                   # one row per image
assert x.shape == (16, 784)
print("Checkpoint 1 passed: flatten gives", tuple(x.shape))

> **Pitfall.** A NumPy image is usually `H, W` or `H, W, C`. A PyTorch model wants `C, H, W`. A silent shape mix-up often still runs and gives garbage.

## 2. Autograd
Take $L(w) = (w\,x - y)^2$ with $x=3$, $y=6$ and $w=1$. By hand: $\partial L/\partial w = 2\,(w x - y)\,x$. 

**TODO 2.** Compute the loss, call `.backward()`, and compare `w.grad` with the hand formula.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
xv, yv = torch.tensor(3.0), torch.tensor(6.0)
# TODO 2a: loss = (w*x - y)^2
loss = None
# TODO 2b: backpropagate (hint: loss.backward())

hand = 2 * (w.item() * xv - yv) * xv
print("autograd:", w.grad.item(), "| by hand:", hand.item())

In [ ]:
assert w.grad is not None and abs(w.grad.item() - hand.item()) < 1e-6
print("Checkpoint 2 passed: gradient =", w.grad.item(), "(negative: increasing w lowers the loss)")

> **Pitfall.** Gradients **accumulate**. Calling `backward()` twice adds the second gradient to the first, so training loops call `optimizer.zero_grad()` every step.

In [ ]:
w.grad.zero_()
(loss_again := (w * xv - yv) ** 2).backward(); (loss_again2 := (w * xv - yv) ** 2).backward()
print("after two backward calls without zeroing:", w.grad.item(), "(twice the gradient)")
w.grad.zero_()

## 3. Gradient descent by hand
We fit a line $y = a x + b$ to noisy points. Loss: mean squared error. The update is $\theta \leftarrow \theta - \eta\,\nabla L$.

**TODO 3.** Complete the loop: forward, loss, backward, update (inside `torch.no_grad()`), zero the gradients.

In [ ]:
set_seed(0)
x = torch.linspace(-1, 1, 60)
y = 2.0 * x + 0.5 + 0.1 * torch.randn(60)
a = torch.zeros((), requires_grad=True); b = torch.zeros((), requires_grad=True)
lr, history = 0.1, []
for step in range(100):
    # TODO 3a: forward pass: pred = a * x + b
    pred = None
    # TODO 3b: mean squared error between pred and y
    loss = None
    # TODO 3c: backward pass
    
    with torch.no_grad():
        # TODO 3d: gradient step on a and b (a -= lr * a.grad ...)
        pass
    a.grad.zero_(); b.grad.zero_()
    history.append(loss.item())
print(f"a = {a.item():.2f} (true 2.0) | b = {b.item():.2f} (true 0.5) | final loss {history[-1]:.4f}")

In [ ]:
assert history[-1] < history[0] * 0.05, "the loss should fall by more than 95%"
assert abs(a.item() - 2.0) < 0.15 and abs(b.item() - 0.5) < 0.15
plt.plot(history); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("MSE"); plt.title("training loss"); plt.show()
print("Checkpoint 3 passed")

> **Reflection.** Change `lr` to 1.5 and to 0.001 (Plus below). What does a loss that goes up tell you? What does a loss that barely moves tell you? Remember this when your CNN does not learn.

> **Plus · learning rate experiment.** Run the loop for `lr` in `[0.001, 0.1, 0.9, 1.1]` and plot the four curves on one figure. Which diverge?

In [ ]:
# [PLUS]
# TODO (Plus): wrap the loop above in a function fit(lr) that returns the loss history;
# plot four learning rates on one figure (clip the loss at 10 for display) and say which diverge
...

## 4. Data: what a `Dataset` and `DataLoader` give you
A **Dataset** returns one example `(image, label)`. A **DataLoader** groups examples into shuffled batches. Below, PneumoniaMNIST (chest X-ray, normal vs pneumonia, 28 × 28). If the download fails (no internet), the cell falls back to random data so the rest runs.

In [ ]:
from torch.utils.data import DataLoader
try:
    import medmnist
    from medmnist import PneumoniaMNIST
    import torchvision.transforms as T
    tf = T.Compose([T.ToTensor()])
    # TODO 4a: load the TRAIN split with transform=tf, download=True, root="data"
    train_ds = None
    n_train = len(train_ds); real = True
except Exception as e:
    print("Download failed, using random data:", repr(e)); real = False
    from torch.utils.data import TensorDataset
    train_ds = TensorDataset(torch.rand(256, 1, 28, 28), torch.randint(0, 2, (256, 1)))
    n_train = len(train_ds)
# TODO 4b: a DataLoader with batch_size=32 and shuffle=True
loader = None
xb, yb = next(iter(loader))
print("real data:", real, "| train size:", n_train, "| batch:", tuple(xb.shape), tuple(yb.shape), "| pixel range:", float(xb.min()), "-", float(xb.max()))

In [ ]:
assert xb.shape == (32, 1, 28, 28) and yb.shape == (32, 1)
if real:
    assert n_train == 4708, f"PneumoniaMNIST train split should have 4708 images, got {n_train}"
    pos = float(np.mean(train_ds.labels)); print(f"class balance: {pos:.0%} pneumonia")
    fig, ax = plt.subplots(1, 6, figsize=(12, 2.4))
    for a_, i in zip(ax, range(6)): a_.imshow(train_ds[i][0][0], cmap="gray"); a_.set_title("pneumonia" if train_ds[i][1][0] else "normal"); a_.axis("off")
    plt.show()
print("Checkpoint 4 passed")

> **Reflection 4.** The cell above printed the share of pneumonia images in the training set. What accuracy does a model that always answers "pneumonia" get? Why is accuracy alone a poor report here? (We build the full answer in NB04 and NB06.)

> **Challenge · same fit with `nn.Module`.** Rewrite section 3 using `torch.nn.Linear(1, 1)`, `nn.MSELoss` and `torch.optim.SGD`. Confirm you get the same slope and intercept (within 0.05).

In [ ]:
# [CHALLENGE]
# TODO (Challenge): model = nn.Linear(1, 1); optimiser SGD lr=0.1; loop: zero_grad, forward, loss, backward, step
...

## Summary
- A tensor has shape, dtype and device; channels come first.
- Autograd gives the gradient; the optimiser uses it; always zero gradients.
- A loss curve is your first diagnostic.
- A `DataLoader` serves shuffled batches from a `Dataset`.

*Next (Day 2):* NB03b (a small network) and NB04 (a baseline CNN).